## Auto Loader — schema inference & evolution modes

Auto Loader infers the schema on the first batch (or from a hint) and **caches it at `cloudFiles.schemaLocation`**. On every later batch it compares the file's schema to the cache; `cloudFiles.schemaEvolutionMode` decides what happens on a new column:

| Mode | On a new column |
|---|---|
| **`addNewColumns`** (default) | Stream **fails**; restart picks up the column, evolves the schema, continues. No silent drift, easy recovery. |
| **`rescue`** | Stream continues; unknown columns land in `_rescued_data`. Nothing fails, nothing lost. |
| **`failOnNewColumns`** | Stream fails permanently until you evolve the schema yourself. Strictest — for regulated data. |
| **`none`** | New columns silently ignored. Only when you fully control upstream. |

**The `_rescued_data` column** captures anything that didn't fit the schema (a new column, an uncastable value) as JSON — so with `rescue` mode **nothing the source emits is ever silently dropped**. Critical for audit.

**Schema hints** — `cloudFiles.schemaHints = 'amount DECIMAL(18,2), is_flagged BOOLEAN'` pin types without writing a full schema.